# Modelo D: fine-tuning de un modelo preentrenado

Ajuste fino de `Helsinki-NLP/opus-mt-en-es`, un Transformer encoder-decoder ya entrenado para inglés → español.
Seguimos el notebook de fine-tuning de T5 del capítulo 15 (`Seq2SeqTrainer`). Probamos dos configuraciones y
nos quedamos con la que da mejor BLEU en validación. Usar GPU (L4).

In [ ]:
REPO_URL = 'https://github.com/EmilianoRL/NLP_Proyecto.git'
REPO_DIR = 'NLP_Proyecto'

import os, sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install -q evaluate sacrebleu sacremoses datasets
if IN_COLAB and not os.path.exists('common'):
    if not os.path.exists(REPO_DIR):
        !git clone -q {REPO_URL}
    %cd {REPO_DIR}
elif not os.path.exists('common') and os.path.exists('../common'):
    %cd ..

In [ ]:
import json
import math
import random
import shutil
import numpy as np
import torch
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from transformers import set_seed

use_gpu = True
device = torch.device('cuda' if use_gpu and torch.cuda.is_available() else 'cpu')

seed = 42
set_seed(seed)
print(f'device: {device.type}')

In [ ]:
from common import checkpoint as ckpt
from common.data import get_splits

SAVE_DIR = ckpt.setup_save_dir()
DATA_DIR = os.path.join(SAVE_DIR, 'data')
FIG_DIR = os.path.join(SAVE_DIR, 'figures')
os.makedirs(FIG_DIR, exist_ok=True)

train_pairs, val_pairs, test_pairs = get_splits(DATA_DIR)

In [ ]:
transformer_name = 'Helsinki-NLP/opus-mt-en-es'
max_source_length = 128
max_target_length = 128
batch_size = 32
num_train_epochs = 4
label_pad_token_id = -100
num_beams = 1          # greedy, igual que en los otros modelos
output_dir = os.path.join(SAVE_DIR, 'modeloD')

## Tokenización

Este modelo usa su propio tokenizador de subpalabras, así que no usamos `clean()` ni nuestro vocabulario:
entra el texto original. `clean()` solo se aplica al final, para calcular BLEU igual que en los otros modelos.

In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset

tokenizer = AutoTokenizer.from_pretrained(transformer_name)

def to_dataset(pairs):
    return Dataset.from_dict({'en': [en for en, es in pairs], 'es': [es for en, es in pairs]})

def tokenize(batch):
    # text_target hace que el español se tokenice con el vocabulario del español
    return tokenizer(
        batch['en'],
        text_target=batch['es'],
        max_length=max_source_length,
        truncation=True,
    )

train_dataset = to_dataset(train_pairs).map(tokenize, batched=True, remove_columns=['en', 'es'])
eval_dataset = to_dataset(val_pairs).map(tokenize, batched=True, remove_columns=['en', 'es'])
train_dataset

In [ ]:
def translate(model, sentences, batch_size=64):
    model.eval()
    predictions = []
    for i in tqdm(range(0, len(sentences), batch_size)):
        encoded = tokenizer(
            sentences[i:i + batch_size],
            max_length=max_source_length,
            truncation=True,
            padding=True,
            return_tensors='pt',
        )
        with torch.no_grad():
            output = model.generate(
                input_ids=encoded.input_ids.to(device),
                attention_mask=encoded.attention_mask.to(device),
                num_beams=num_beams,
                max_length=max_target_length,
            )
        predictions += tokenizer.batch_decode(output, skip_special_tokens=True)
    return predictions

## Modelo sin fine-tuning

Primero vemos qué tan bien traduce el modelo tal como viene. OPUS incluye datos de Tatoeba, así que es probable que
ya haya visto oraciones parecidas (o iguales) a las nuestras.

In [ ]:
from transformers import AutoModelForSeq2SeqLM
from common.metrics import corpus_bleu

val_sources = [en for en, es in val_pairs]
val_references = [es for en, es in val_pairs]

model = AutoModelForSeq2SeqLM.from_pretrained(transformer_name).to(device)
n_params_total = sum(p.numel() for p in model.parameters())
print(f'parámetros: {n_params_total:,}')

val_bleu = {'sin fine-tuning': corpus_bleu(translate(model, val_sources), val_references)}
print('BLEU val sin fine-tuning:', round(val_bleu['sin fine-tuning'], 2))

## Configuraciones

Las dos usan lr = 5e-5 con warmup (10% de los pasos) y weight decay de 0.01. La diferencia:
* `config1`: se ajustan todas las capas.
* `config2`: se congela el encoder (incluye los embeddings, que el modelo comparte con el decoder) y solo se ajusta el decoder.

Cada época se guarda un checkpoint en Drive. Si se corta la sesión, al volver a correr sigue desde el último
(`get_last_checkpoint`, como en el notebook de clase). Al terminar se guarda la mejor época según `val_loss`
y se borran los checkpoints intermedios para no llenar Drive.

In [ ]:
configs = {
    'config1': {'learning_rate': 5e-5, 'weight_decay': 0.01, 'freeze_encoder': False},
    'config2': {'learning_rate': 5e-5, 'weight_decay': 0.01, 'freeze_encoder': True},
}

In [ ]:
from transformers import DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer
from transformers.trainer_utils import get_last_checkpoint

def train_config(name, cfg):
    run_dir = os.path.join(output_dir, name)
    final_dir = os.path.join(run_dir, 'final')
    if os.path.exists(final_dir):
        print(f'{name} ya está entrenado')
        return

    set_seed(seed)
    model = AutoModelForSeq2SeqLM.from_pretrained(transformer_name)
    if cfg['freeze_encoder']:
        for p in model.get_encoder().parameters():
            p.requires_grad = False
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'{name}: parámetros entrenables {n_params:,}')

    data_collator = DataCollatorForSeq2Seq(
        tokenizer,
        model=model,
        label_pad_token_id=label_pad_token_id,
    )
    steps_per_epoch = math.ceil(len(train_dataset) / batch_size)
    training_args = Seq2SeqTrainingArguments(
        output_dir=run_dir,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        learning_rate=cfg['learning_rate'],
        weight_decay=cfg['weight_decay'],
        warmup_steps=int(0.1 * steps_per_epoch * num_train_epochs),
        num_train_epochs=num_train_epochs,
        eval_strategy='epoch',
        save_strategy='epoch',
        logging_strategy='epoch',
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model='eval_loss',
        report_to='none',
        seed=seed,
    )
    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        processing_class=tokenizer,
        data_collator=data_collator,
    )

    last_checkpoint = None
    done_epochs = 0
    if os.path.isdir(run_dir):
        last_checkpoint = get_last_checkpoint(run_dir)
    if last_checkpoint is not None:
        with open(os.path.join(last_checkpoint, 'trainer_state.json')) as f:
            done_epochs = json.load(f)['epoch']
        print(f'Checkpoint encontrado, se sigue desde {last_checkpoint}')

    train_result = trainer.train(resume_from_checkpoint=last_checkpoint)
    trainer.save_model(final_dir)       # mejor época según val_loss

    log = trainer.state.log_history
    info = {
        'history': {'train_loss': [h['loss'] for h in log if 'loss' in h],
                    'val_loss': [h['eval_loss'] for h in log if 'eval_loss' in h]},
        # train_runtime incluye la evaluación en val de cada época
        'epoch_time': train_result.metrics['train_runtime'] / (num_train_epochs - done_epochs),
        'trainable_params': n_params,
    }
    with open(os.path.join(run_dir, 'info.json'), 'w') as f:
        json.dump(info, f, indent=2)

    for d in os.listdir(run_dir):
        if d.startswith('checkpoint-'):
            shutil.rmtree(os.path.join(run_dir, d))

In [ ]:
del model
torch.cuda.empty_cache()

for name, cfg in configs.items():
    train_config(name, cfg)

In [ ]:
infos = {}
for name in configs:
    with open(os.path.join(output_dir, name, 'info.json')) as f:
        infos[name] = json.load(f)

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, name in zip(axes, configs):
    h = infos[name]['history']
    x = np.arange(len(h['train_loss'])) + 1
    ax.plot(x, h['train_loss'])
    ax.plot(x, h['val_loss'])
    ax.legend(['train', 'val'])
    ax.set_xlabel('epoch')
    ax.set_ylabel('loss')
    ax.set_title(f'Modelo D ({name})')
    ax.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'curvas_modeloD.png'), dpi=150)
plt.show()

## Selección con validación

In [ ]:
import pandas as pd

for name in configs:
    model = AutoModelForSeq2SeqLM.from_pretrained(os.path.join(output_dir, name, 'final')).to(device)
    val_bleu[name] = corpus_bleu(translate(model, val_sources), val_references)
    del model
    torch.cuda.empty_cache()

summary = pd.DataFrame({
    'BLEU val': {k: round(v, 2) for k, v in val_bleu.items()},
    'mejor val_loss': {name: round(min(infos[name]['history']['val_loss']), 4) for name in configs},
    'parámetros entrenables': {name: infos[name]['trainable_params'] for name in configs},
    'tiempo/época (s)': {name: round(infos[name]['epoch_time']) for name in configs},
})
best_name = max(configs, key=lambda k: val_bleu[k])
print('mejor configuración:', best_name)
summary

## Evaluación en test

Solo con la configuración elegida. La pérdida y la perplejidad se calculan sobre subpalabras, así que no se pueden
comparar directamente con las de los modelos A, B y C (que son por palabra).

In [ ]:
model = AutoModelForSeq2SeqLM.from_pretrained(os.path.join(output_dir, best_name, 'final')).to(device)
test_dataset = to_dataset(test_pairs).map(tokenize, batched=True, remove_columns=['en', 'es'])

eval_args = Seq2SeqTrainingArguments(output_dir='/tmp/eval_modeloD', per_device_eval_batch_size=batch_size,
                                     report_to='none')
evaluator = Seq2SeqTrainer(model=model, args=eval_args, processing_class=tokenizer,
                           data_collator=DataCollatorForSeq2Seq(tokenizer, model=model,
                                                                label_pad_token_id=label_pad_token_id))
test_loss = evaluator.evaluate(test_dataset)['eval_loss']
print(f'test loss: {test_loss:.4f} | perplejidad: {math.exp(test_loss):.2f}')

In [ ]:
from common.metrics import bleu_report

sources = [en for en, es in test_pairs]
references = [es for en, es in test_pairs]
predictions = translate(model, sources)
bleu = bleu_report(sources, predictions, references)
bleu

In [ ]:
from common.metrics import measure_latency, gpu_name, save_results

def translate_one(sentence):
    encoded = tokenizer([sentence], return_tensors='pt')
    with torch.no_grad():
        output = model.generate(input_ids=encoded.input_ids.to(device),
                                attention_mask=encoded.attention_mask.to(device),
                                num_beams=num_beams, max_length=max_target_length)
    return tokenizer.decode(output[0], skip_special_tokens=True)

latency = measure_latency(translate_one, sources[:300], device)
print(f'latencia: {latency:.1f} ms por oración ({gpu_name()})')

results = {
    'modelo': 'D',
    'configuracion': best_name,
    'parametros': infos[best_name]['trainable_params'],
    'parametros_totales': n_params_total,
    'tiempo_epoca_s': infos[best_name]['epoch_time'],
    'epocas': len(infos[best_name]['history']['train_loss']),
    'gpu': gpu_name(),
    'test_loss': test_loss,
    'test_ppl': math.exp(test_loss),
    'bleu_global': bleu['global'],
    'bleu_media': bleu['media'],
    'bleu_larga': bleu['larga'],
    'latencia_ms': latency,
    'predicciones': predictions,
}
save_results(SAVE_DIR, 'modeloD', results)

Como referencia, el BLEU en test del modelo sin fine-tuning:

In [ ]:
base_model = AutoModelForSeq2SeqLM.from_pretrained(transformer_name).to(device)
base_predictions = translate(base_model, sources)
base_bleu = bleu_report(sources, base_predictions, references)
save_results(SAVE_DIR, 'modeloD_sin_finetuning', {
    'modelo': 'D sin fine-tuning',
    'bleu_global': base_bleu['global'],
    'bleu_media': base_bleu['media'],
    'bleu_larga': base_bleu['larga'],
    'predicciones': base_predictions,
})
pd.DataFrame({'sin fine-tuning': base_bleu, f'fine-tuning ({best_name})': bleu}).round(2)

## Ejemplos

In [ ]:
pd.set_option('display.max_colwidth', 100)

# las mismas oraciones que en el notebook del Modelo A
rng = random.Random(0)
idx = rng.sample(range(len(test_pairs)), 4)
idx.append(next(i for i, en in enumerate(sources) if len(en.split()) > 20))
pd.DataFrame({'inglés': [sources[i] for i in idx],
              'referencia': [references[i] for i in idx],
              'modelo D': [predictions[i] for i in idx]})